# Direct-current studies

`f = 0` is an ordinary entry in `Network.frequencies`, and a direct-current
study needs nothing beyond it. What needs care at zero frequency is not the
solver — $Y(f)\,\underline u(f) = \underline i(f)$ has no frequency dependence
of its own and is exact at zero — but what an impedance *formula* means there:

| what | example | what `groundinsight` does at 0 Hz |
|---|---|---|
| a **removable singularity** | Carson's earth-return term `2*pi*f*log(658*sqrt(rho/f)/GMR)` | evaluates the limit, the DC resistance |
| a **finite reactance** | `(0.25 + I*0.6)*l` | drops the reactance and warns |
| an impedance of **exactly zero** | a purely inductive bond, `I*2*pi*f*L*l` | a finite stand-in, announced by a warning |
| a real part of **minus infinity** | `log(f)` without a factor `f` | refuses the formula |

This notebook measures each case against references that do **not** come
from `groundinsight` — a node-merged $2\times2$ system and an open-circuit
$3\times3$ system solved with NumPy, the closed-form limit of Carson's term
and the linearity of the network — and it puts a number on what the old
workaround, entering `f = 0.1 Hz` instead of zero, costs.

Background: [Direct current (`f = 0`)](../../explanation/dc/).

In [ ]:
import warnings

import numpy as np
import polars as pl

import groundinsight as gi
from groundinsight import waveforms

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(90)
print(f"groundinsight {gi.__version__}")

## The reference network

Three buses in a chain, $A - B - C$:

| bus | what it is | $Z_B$ |
|---|---|---|
| `A` | earthing grid of a 110 kV substation | $0.8\ \Omega$ |
| `B` | tower footing / joint bay half way along the line | $12\ \Omega$ |
| `C` | earthing grid of the remote station | $3.5\ \Omega$ |

A direct current of 1 kA is driven through the earthing system between `A`
and `C` — stray direct current from a nearby DC traction return, the electrode
line of an HVDC link, a cathodic-protection installation, or the quasi-DC
neutral current of a geomagnetic disturbance. The question is the usual one:
how much of it takes the earthing conductor and how much takes the soil, and
what earth potential rise is left at each station.

The same chain is used in two configurations:

* a **line configuration** for sections 1 to 3, where both spans are 10 km of
  the earthing conductor under test;
* a **bond configuration** for sections 4 to 6, where `A-B` is a short
  bonding conductor with no resistance at all and `B-C` is an ordinary
  $0.35\ \Omega$ conductor.

Two buses are not enough for the second one: an ideal bond between the source
bus and the fault bus merges them into a single node at which the injection
`+I` and `-I` cancels, so every potential is zero and the treatment of the
bond is invisible.

In the sign frame of `groundinsight` the source current enters the earthing
network at the source bus `A` and leaves it at the fault bus `C`, so the
potentials come out positive at `A` and negative at `C`; the physical fault
picture, with the earth current entering the soil at the fault, is the
reverse ([Signs and phase angles](../../reference/conventions/#signs-and-phase-angles)).
Every magnitude below is independent of that choice.

In [ ]:
Z_A, Z_B, Z_C = 0.8, 12.0, 3.5   # Ohm    earthing impedance of the three buses
RHO = 100.0                      # Ohm m  soil resistivity
I_DC = 1000.0                    # A      direct current entering at A

# Two spellings of the same 0.25 Ohm/km earthing conductor. The first lets the
# reactance vanish at DC by itself, the second does not.
CARSON = "(0.25 + I*2*pi*f*2e-7*log(658*sqrt(rho/f)/0.01)) * l"
CONSTANT_X = "(0.25 + I*0.6) * l"
NO_COUPLING = "(rho*0 + 0.0 + I*f*0) * l"


def build(name, ab_formula, bc_formula, *, frequencies=(0.0,),
          ab_length=10.0, bc_length=10.0, voltage_source=False):
    "Chain A -- B -- C with ``I_DC`` entering at A and the fault at C."
    freqs = sorted(float(f) for f in frequencies)
    bus_types = {
        label: gi.BusType(
            name=f"{name}_bt_{label}",
            description=f"Earthing system at {label}",
            system_type="Substation",
            voltage_level=110.0,
            impedance_formula=f"rho*0 + {z!r} + I*f*0",
        )
        for label, z in (("A", Z_A), ("B", Z_B), ("C", Z_C))
    }
    ab_type = gi.BranchType(name=f"{name}_ab", grounding_conductor=True,
                            self_impedance_formula=ab_formula,
                            mutual_impedance_formula=NO_COUPLING)
    bc_type = gi.BranchType(name=f"{name}_bc", grounding_conductor=True,
                            self_impedance_formula=bc_formula,
                            mutual_impedance_formula=NO_COUPLING)
    net = gi.create_network(name=name, frequencies=freqs, description="DC earth-current study")
    for label in ("A", "B", "C"):
        gi.create_bus(name=label, type=bus_types[label], specific_earth_resistance=RHO, network=net)
    gi.create_branch(name="A-B", type=ab_type, from_bus="A", to_bus="B", length=ab_length, network=net)
    gi.create_branch(name="B-C", type=bc_type, from_bus="B", to_bus="C", length=bc_length, network=net)
    if voltage_source:
        gi.create_voltage_source(
            name="dc", bus="A",
            voltage={f: I_DC + 0.0j for f in freqs},
            source_impedance={f: 0.5 + 0.0j for f in freqs},
            network=net,
        )
    else:
        gi.create_source(name="dc", bus="A", values={f: I_DC + 0.0j for f in freqs}, network=net)
    gi.create_fault(name="F", bus="C", network=net)
    return net


def epr(net, freq=0.0, fault="F"):
    "Earth potential rise per bus as complex phasors; at DC they are real."
    return {b.name: complex(b.uepr_freq[freq].real, b.uepr_freq[freq].imag)
            for b in net.results[fault].buses}


def evaluate(formula, frequencies, length=10.0):
    "The self impedance of one branch with this formula, per frequency."
    probe_net = gi.create_network(name="probe", frequencies=sorted(frequencies))
    station = gi.BusType(name="probe_station", system_type="Substation", voltage_level=110.0,
                         impedance_formula="1.0")
    for bus in ("P", "Q"):
        gi.create_bus(name=bus, type=station, specific_earth_resistance=RHO, network=probe_net)
    probe = gi.BranchType(name="probe", grounding_conductor=True, self_impedance_formula=formula,
                          mutual_impedance_formula=NO_COUPLING)
    branch = gi.create_branch(name="P-Q", type=probe, from_bus="P", to_bus="Q", length=length,
                              specific_earth_resistance=RHO, network=probe_net)
    return {f: complex(z.real, z.imag) for f, z in branch.self_impedance.items()}

## 1  What the `f = 0.1 Hz` workaround costs

The same study four times: two spellings of the conductor, each at 0.1 Hz and
at 0 Hz. $Z_E$ is the earthing impedance at the fault bus `C`.

In [ ]:
rows = []
for spelling, formula in (("I*2*pi*f*L (Carson)", CARSON), ("constant I*X", CONSTANT_X)):
    for freq in (0.1, 0.0):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", gi.DCLimitWarning)  # section 2 shows it
            net = build(f"cost_{len(rows)}", formula, formula, frequencies=(freq,))
            gi.run_fault(net, fault="F")
        z = net.branches["A-B"].self_impedance[freq]
        rows.append({
            "spelling": spelling,
            "f_Hz": freq,
            "Z_self(A-B)_Ohm": f"{z.real:.4f}{z.imag:+.4g}j",
            "|EPR(A)|_V": abs(epr(net, freq)["A"]),
            "|Z_E|_Ohm": float(net.res_all_impedances()["grounding_impedance_Ohm"][0]),
        })

pl.DataFrame(rows)

In [ ]:
carson = {r["f_Hz"]: r["|EPR(A)|_V"] for r in rows if r["spelling"].startswith("I*2")}
const = {r["f_Hz"]: r["|EPR(A)|_V"] for r in rows if r["spelling"].startswith("constant")}

print(f"Carson       0.1 Hz -> {carson[0.1]:.9f} V    0 Hz -> {carson[0.0]:.9f} V"
      f"    rel. difference {abs(carson[0.1] - carson[0.0]) / abs(carson[0.0]):.2e}")
print(f"constant I*X 0.1 Hz -> {const[0.1]:.4f} V         0 Hz -> {const[0.0]:.4f} V"
      f"         rel. difference {abs(const[0.1] - const[0.0]) / abs(const[0.0]):.1%}")

For the Carson spelling the workaround is accurate — eleven digits. For the
constant-reactance spelling, which is by far the more common one in practice,
0.1 Hz keeps the $6\ \Omega$ of reactance per span that does not exist at DC
and reports an earth potential rise **more than 50 % too high**, and an
earthing impedance 40 % too high. The reactance is constant, so making the
frequency smaller does not make the error smaller: `f = 1e-6 Hz` is exactly as
wrong as `f = 0.1 Hz`.

## 2  A finite reactance has no reading at DC

At zero frequency a reactance can only vanish ($j\omega L \to 0$) or be
infinite ($1/(j\omega C) \to \infty$). A finite non-zero one is a statement
about nothing, so the 0 Hz bin takes the real part, drops the reactance and
says so. Every other frequency is untouched.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    z_const = evaluate(CONSTANT_X, [0.0, 50.0])
    z_carson = evaluate(CARSON, [0.0, 50.0])

for freq in (0.0, 50.0):
    print(f"f = {freq:>4} Hz   constant I*X -> {z_const[freq]!r:28}   Carson -> {z_carson[freq]!r}")

print()
for w in caught:
    print(f"{w.category.__name__} ({w.message.code}):\n  {w.message}\n")

Only the constant-reactance formula warns. Two things about that warning are
deliberate:

* it quotes the **ratio** $X/R$ rather than the two values, because the ratio
  is length-invariant and therefore identical for every branch sharing a
  `BranchType` — which lets Python's default warning filter collapse a
  hundred-branch network into a single line instead of a hundred;
* it names the **remedy**: write the reactance as `I*2*pi*f*L` and it vanishes
  at DC by itself, with nothing dropped and nothing warned about.

Ten branches of one type and ten different lengths, under Python's default
warning filter:

In [ ]:
with warnings.catch_warnings(record=True) as shown:
    warnings.simplefilter("default")
    chain = gi.create_network(name="dedup", frequencies=[0.0])
    station = gi.BusType(name="dedup_station", system_type="Substation", voltage_level=110.0,
                         impedance_formula="rho*0 + 2.0 + I*f*0")
    line = gi.BranchType(name="dedup_line", grounding_conductor=True,
                         self_impedance_formula=CONSTANT_X, mutual_impedance_formula=NO_COUPLING)
    for k in range(11):
        gi.create_bus(name=f"N{k}", type=station, specific_earth_resistance=RHO, network=chain)
    for k in range(10):
        gi.create_branch(name=f"L{k}", type=line, from_bus=f"N{k}", to_bus=f"N{k + 1}",
                         length=0.5 + 0.3 * k, network=chain)

print(f"{len(chain.branches)} branches, {len(shown)} warning shown")

The fallback is for impedance formulas only. `R_formula`, `L_formula` and
`C_formula`, which feed the state-space transient solver, are real at every
frequency by contract, so a complex value there is reported as an error
instead of being repaired.

## 3  Carson's earth-return term at zero frequency

$$
\underline Z(f) = \Big(R' + j\,\omega\,2\cdot10^{-7}
        \ln\frac{658\sqrt{\rho/f}}{\text{GMR}}\Big)\, l ,
\qquad \omega = 2\pi f .
$$

At $f = 0$ this is $0\cdot\infty$: $\omega$ vanishes linearly while the
logarithm diverges only logarithmically, so the limit exists and is $0$ — the
conductor tends to its DC resistance, which is the one number a DC study is
about. Floating point renders the limit as `NaN`; the evaluator finds the
limit instead of reporting a formula error.

In [ ]:
approach = [1.0, 1e-1, 1e-2, 1e-3, 1e-4, 1e-6, 1e-8, 0.0]
z = evaluate(CARSON, approach)

pl.DataFrame([
    {"f_Hz": f, "Re Z / Ohm": z[f].real, "Im Z / Ohm": z[f].imag,
     "|Im/Re|": abs(z[f].imag / z[f].real)}
    for f in approach
])

The reactance falls by one decade per decade of frequency and is *exactly*
zero at 0 Hz, where the real part is the DC resistance $0.25\ \Omega/\text{km}
\times 10\ \text{km} = 2.5\ \Omega$. The limit is found numerically, by
approaching zero on the decade sequence `1e-6, 1e-7, 1e-8 Hz` and comparing
two consecutive **absolute** differences: shrinking means a removable
singularity, growing means a true pole. The criterion has to be absolute — a
relative one cannot classify a formula whose limit is zero, because there the
relative change per decade stays at 90 % forever.

In [ ]:
for label, formula in (
    ("inductance  I*2*pi*f*L", "rho*0 + I*2*pi*f*2.5e-3"),
    ("capacitance 1/(I*w*C) ", "rho*0 + 1/(I*2*pi*f*1e-6)"),
):
    probe = evaluate(formula, [1e-6, 1e-7, 1e-8], length=1.0)
    mags = [abs(probe[f]) for f in (1e-6, 1e-7, 1e-8)]
    d1, d2 = abs(mags[1] - mags[0]), abs(mags[2] - mags[1])
    verdict = "convergent -> limit" if d2 < d1 else "divergent -> pole"
    print(f"{label}   d1 = {d1:9.3e}   d2 = {d2:9.3e}   {verdict}")

print()
print("at 0 Hz:")
for label, formula in (("inductance ", "rho*0 + I*2*pi*f*2.5e-3"),
                       ("capacitance", "rho*0 + 1/(I*2*pi*f*1e-6)")):
    print(f"  {label} -> {evaluate(formula, [0.0], length=1.0)[0.0]!r}")

The tie-break is biased towards *convergent* on purpose. Mistaking a pole for
a limit yields a very large finite impedance, which behaves almost like the
open circuit it should have been; the opposite mistake would silently
disconnect a real earthing conductor. A formula that is `NaN` on the approach
as well is a genuine failure and raises, with a message that names the
formula, the frequency and the parameters.

In [ ]:
station = gi.BusType(name="nan_station", system_type="Substation", voltage_level=110.0,
                     impedance_formula="rho + I*2*pi*f*1e-3")
try:
    gi.create_bus(name="X", type=station, specific_earth_resistance=float("nan"),
                  network=gi.create_network(name="dc", frequencies=[0.0]))
except ValueError as exc:
    print(f"ValueError: {exc}")

A real part of **minus infinity** is none of these. `log(f)` without a factor
$f$ runs to $-\infty$ at 0 Hz: the limit of negative resistances, not the open
circuit $+\infty$ of a pole. It would read as an open circuit, because
$1/(-\infty) = 0$, so it is refused wherever an impedance is inverted:

In [ ]:
broken = gi.BusType(name="broken", system_type="Substation", voltage_level=110.0,
                    impedance_formula="2 + log(f)")
try:
    gi.create_bus(name="X", type=broken, network=gi.create_network(name="dc", frequencies=[0.0]))
except ValueError as exc:
    print(f"ValueError: {str(exc)[:240]} ...")

## 4  An ideal bond is a short circuit, and a short circuit has no admittance

Now the bond configuration: `A-B` is a 1 km bonding conductor written as pure
inductance, with no resistance at all, and `B-C` is an ordinary
$0.35\ \Omega$ conductor. At DC the bond is an **exact** zero — correct
physics, and a number the nodal formulation cannot invert.

The reference is not `groundinsight`: an ideal bond means the two buses are
one node, so the correct DC answer comes from a hand-solved $2\times2$ system
in which $A$ and $B$ are merged and their electrodes are in parallel. The
second reference is what dropping the element would give — modelling the
short circuit as a disconnection.

In [ ]:
R_BC = 0.35        # Ohm   ordinary conductor B-C over 1 km
L_BOND = 1.2e-3    # H     bonding conductor A-B over 1 km, no resistance
BOND = f"(rho*0 + 0.0 + I*2*pi*f*{L_BOND!r}) * l"
ORDINARY = f"(rho*0 + {R_BC!r} + I*2*pi*f*4e-4) * l"


def merged_reference():
    "DC solution with the ideal bond merged into a single node {A, B}."
    g = 1.0 / R_BC
    Y = np.array([[1 / Z_A + 1 / Z_B + g, -g], [-g, 1 / Z_C + g]])
    u = np.linalg.solve(Y, np.array([I_DC, -I_DC]))
    return {"A": u[0], "B": u[0], "C": u[1]}


def open_reference():
    "DC solution with the bond dropped from the matrix (a short modelled as open)."
    g = 1.0 / R_BC
    Y = np.array([
        [1 / Z_A, 0.0, 0.0],
        [0.0, 1 / Z_B + g, -g],
        [0.0, -g, 1 / Z_C + g],
    ])
    u = np.linalg.solve(Y, np.array([I_DC, 0.0, -I_DC]))
    return {"A": u[0], "B": u[1], "C": u[2]}


with warnings.catch_warnings(record=True) as caught_bond:
    warnings.simplefilter("always")
    bond_net = build("bond", BOND, ORDINARY, frequencies=(0.0,), ab_length=1.0, bc_length=1.0)
    gi.run_fault(bond_net, fault="F")

got = {bus: value.real for bus, value in epr(bond_net).items()}
ref, opened = merged_reference(), open_reference()

pl.DataFrame([
    {
        "bus": b,
        "groundinsight_V": got[b],
        "merged reference_V": ref[b],
        "rel. error": abs(got[b] - ref[b]) / abs(ref[b]),
        "bond dropped_V": opened[b],
        "factor off": opened[b] / ref[b],
    }
    for b in ("A", "B", "C")
])

In [ ]:
for w in caught_bond:
    print(f"{w.category.__name__} ({w.message.code}):\n  {w.message}\n")

The substitution agrees with the merged reference to better than $10^{-7}$
relative at every bus.
Dropping the element instead would be wrong by an order of magnitude at every
bus and would **reverse the sign at bus B**: bus `A` would be cut off from the
network and report its own electrode ($1000\ \text{A} \times 0.8\ \Omega =
800\ \text{V}$) instead of the parallel combination.

The stand-in is $\sqrt{\varepsilon_\text{mach}}\,Z_\text{min} = 1.4901\cdot
10^{-8}\,Z_\text{min}$, with $Z_\text{min}$ the smallest finite non-zero
impedance magnitude in the network at that frequency, and it is applied at the
0 Hz bin only. Nothing else in the study is touched.

In [ ]:
print(f"{'R_bond / Ohm':>14}  {'EPR(A) / V':>14}  {'rel. to merged':>15}")
print("-" * 48)
for r_bond in (1e-2, 1e-4, 1e-6, 1e-8):
    n = build(f"explicit_{r_bond:g}", f"(rho*0 + {r_bond!r} + I*f*0) * l",
              ORDINARY, frequencies=(0.0,), ab_length=1.0, bc_length=1.0)
    gi.run_fault(n, fault="F")
    e = epr(n)["A"].real
    print(f"{r_bond:14.0e}  {e:14.6f}  {abs(e - ref['A']) / abs(ref['A']):15.3e}")
print(f"{'substitution':>14}  {got['A']:14.6f}  {abs(got['A'] - ref['A']) / abs(ref['A']):15.3e}")

Giving the bond a small but honest resistance converges to the same answer
from the other side: the deviation shrinks in proportion to the resistance,
about $3.5\,R_\text{bond}/\Omega$ relative in this network.

**When to merge the two buses instead.** The exact model of an ideal bond is
one node, not two, and the stand-in is an approximation of that merge. Prefer
the merge — a single bus whose impedance is the parallel combination of the
two electrodes — whenever the *difference* in earth potential rise across the
bond is part of what you report (across an ideal bond it is zero, and whatever
the stand-in produces there is numerical residue), or when the bonded buses
carry impedances many orders of magnitude away from $Z_\text{min}$. For the
usual question — the earth potential rise of the bonded group and the current
leaving it — the substitution needs no change to the model.

## 5  Every transient run contains DC

An `rfft` frequency grid always contains a 0 Hz bin, so **every** transient
run evaluates every formula at zero frequency, whether or not the study is
about DC. The DC component of the time-domain answer is the mean of $u(t)$
over a full FFT window, which is exactly the 0 Hz bin — so the stationary
reference above applies unchanged.

In [ ]:
tr_net = build("bond_transient", BOND, ORDINARY, frequencies=(50.0,), ab_length=1.0, bc_length=1.0)
study = gi.TransientStudy(tr_net, fault="F")
study.set_source_waveform("dc", waveforms.step(amplitude=I_DC, t_on=0.0))
study.set_observation(buses=["A", "B", "C"], branches=["A-B", "B-C"])

with warnings.catch_warnings(record=True) as caught_tr:
    warnings.simplefilter("always")
    res = study.solve(t_end=0.04, dt=1e-4, solver="fft")

dc_bin = {b: float(np.mean(np.asarray(v))) for b, v in res.epr_t.items()}
i_bin = {b: float(np.mean(np.asarray(v))) for b, v in res.i_branch_t.items()}

print(pl.DataFrame([
    {"bus": b, "DC bin of u(t)_V": dc_bin[b], "merged reference_V": ref[b],
     "rel. error": abs(dc_bin[b] - ref[b]) / abs(ref[b])}
    for b in ("A", "B", "C")
]))
print()
print("largest difference to the stationary solve of section 4: "
      f"{max(abs(dc_bin[b] - got[b]) for b in ('A', 'B', 'C')):.2e} V")
print()
print("DC component of the branch currents / A:", {k: round(v, 3) for k, v in i_bin.items()})
print()
for w in caught_tr:
    print(f"{w.category.__name__} ({getattr(w.message, 'code', '')}): {str(w.message)[:110]} ...")

The transient run applies the same stand-in in its own 0 Hz bin and lands on
the stationary answer. The `TRANSIENT_MODEL` warning is expected for this
input: it is meant for a response that should have died away by the end of
the window, and a step that stays on never does. For the FFT solver a
constant is exactly periodic, so the result is not affected. The DC
components of the branch currents are physical currents from `from_bus` to
`to_bus`: $923.9\ \text{A}$ in `B-C` is $(u_B - u_C)/0.35\ \Omega$ of the
merged reference, and `A-B` carries $4.8\ \text{A}$ more: the current
that enters the soil through the electrode at `B`, $u_B / 12\ \Omega$.

The state-space solver is the independent cross-check of this path. It
integrates the lumped fields `R_self_formula` and `L_self_formula` in time,
keeps the bond as the inductance it is and never inverts an impedance, so it
needs no stand-in. Its settled DC state:

In [ ]:
def lumped_bond_network():
    "The bond configuration with the lumped fields the state-space solver integrates."
    lumped = gi.create_network(name="bond_state_space", frequencies=[50.0])
    for label, z in (("A", Z_A), ("B", Z_B), ("C", Z_C)):
        bus_type = gi.BusType(name=f"ss_{label}", system_type="Substation", voltage_level=110.0,
                              impedance_formula=f"rho*0 + {z!r} + I*f*0", R_formula=f"rho*0 + {z!r}")
        gi.create_bus(name=label, type=bus_type, specific_earth_resistance=RHO, network=lumped)
    bond = gi.BranchType(name="ss_bond", grounding_conductor=True, self_impedance_formula=BOND,
                         mutual_impedance_formula=NO_COUPLING,
                         R_self_formula="0.0*l", L_self_formula=f"{L_BOND!r}*l")
    ordinary = gi.BranchType(name="ss_ordinary", grounding_conductor=True,
                             self_impedance_formula=ORDINARY, mutual_impedance_formula=NO_COUPLING,
                             R_self_formula=f"{R_BC!r}*l", L_self_formula="4e-4*l")
    gi.create_branch(name="A-B", type=bond, from_bus="A", to_bus="B", length=1.0, network=lumped)
    gi.create_branch(name="B-C", type=ordinary, from_bus="B", to_bus="C", length=1.0, network=lumped)
    gi.create_source(name="dc", bus="A", values={50.0: I_DC}, network=lumped)
    gi.create_fault(name="F", bus="C", network=lumped)
    return lumped


ss_study = gi.TransientStudy(lumped_bond_network(), fault="F")
ss_study.set_source_waveform("dc", waveforms.step(amplitude=I_DC, t_on=0.0))
ss_study.set_observation(buses=["A", "B", "C"])
settled = {b: float(np.asarray(v)[-1])
           for b, v in ss_study.solve(t_end=0.05, dt=1e-5, solver="state_space").epr_t.items()}
for b in ("A", "B", "C"):
    print(f"{b}: state space {settled[b]:12.6f} V   merged reference {ref[b]:12.6f} V"
          f"   rel. error {abs(settled[b] - ref[b]) / abs(ref[b]):.1e}")
print(f"\nu_A - u_B: state space {settled['A'] - settled['B']:.1e} V,"
      f" stationary solve with the stand-in {got['A'] - got['B']:.1e} V")

The state-space solver lands on the merged reference and keeps `A` and `B`
at one potential, as an ideal bond must. The stand-in of the stationary and
FFT solves leaves a few microvolts between them — the numerical residue the
warning refers to when it recommends a single bus for an exact result.

## 6  A true pole is still an open circuit

The substitution must not touch a capacitance. A series capacitance really is
an open circuit at DC, $1/(j\omega C) \to \infty$ says so, and the solver
already reads infinity as an open circuit. The expected answer is therefore
the *open-circuit* reference, not the merged one — bus `A` reduced to its own
electrode.

In [ ]:
CAP_BOND = "(rho*0 + 1/(I*2*pi*f*1e-9)) * l"

with warnings.catch_warnings(record=True) as caught_cap:
    warnings.simplefilter("always")
    cap_net = build("cap_bond", CAP_BOND, ORDINARY, frequencies=(0.0,), ab_length=1.0, bc_length=1.0)
    gi.run_fault(cap_net, fault="F")

z_cap = cap_net.branches["A-B"].self_impedance[0.0]
print("Z_self(A-B) at 0 Hz     :", complex(z_cap.real, z_cap.imag))
print(f"EPR(A)                  : {epr(cap_net)['A'].real:.6f} V")
print(f"open-circuit reference  : {open_reference()['A']:.6f} V")
print("warnings                :", [w.category.__name__ for w in caught_cap] or "none")

## 7  What a source and a fault need at DC

* **Current source** — `values` needs a `0.0` key. Without it the source
  contributes nothing at DC, announced once per run (`InputWarning`, code
  `SOURCE_FREQUENCY_GAP`). Give it a real number: at zero frequency the
  imaginary part is not a phase shift.
* **Voltage source** — `voltage` and `source_impedance` both need a `0.0`
  entry; a network solved at 0 Hz without one is rejected at the solve. The
  `0.0` entry of `source_impedance` is the DC resistance of the source: the
  values are numbers, not formulas, so nothing drops a reactance there for
  you, and zero is rejected when the source is built.
* **Fault** — `scalings` does not need a `0.0` key: a frequency without a key
  is scaled with 1.0, so the sources contribute in full at DC. Give
  `0.0: 0.0` to keep the DC bin out of a fault. A mapping that names some
  network frequencies but not others is announced once per run
  (`InputWarning`, code `SCALINGS_INCOMPLETE`).
* **IEC 60909 characteristics** — `i_k_a`, `r_to_x` and `kappa` of
  `create_source` describe the DC offset and the thermal equivalent of an
  *alternating* short-circuit current. They are metadata that the solve does
  not use, and they have no meaning in a pure DC study; leave them unset
  there.
* **Mixed studies** — `frequencies=[0.0, 50.0]` is legal, and the two do not
  interact: the DC handling is confined to the 0 Hz bin.

In [ ]:
mixed = build("mixed", CARSON, CARSON, frequencies=(0.0, 50.0))
gi.run_fault(mixed, fault="F")

ac_only = build("ac_only", CARSON, CARSON, frequencies=(50.0,))
gi.run_fault(ac_only, fault="F")

print("50 Hz answer identical to an AC-only study, bit for bit:",
      epr(mixed, 50.0) == epr(ac_only, 50.0))
print()
print(mixed.res_buses(fault="F").select("bus_name", "frequency_Hz", "is_rms", "EPR_V",
                                         "EPR_degree", "I_bus_A"))

A voltage source is the second way to drive the study, and at DC it is worth
seeing what it does. `groundinsight` closes the Thévenin loop between the
source bus and the fault bus, so the current that ends up flowing is

$$
I = \frac{U}{Z_{\text{source}} + Z_{\text{loop}}},
\qquad
Z_{\text{loop}} = \frac{u_A - u_C}{I_{\text{DC}}}
$$

with $u_A$, $u_C$ taken from the current-source study above. Since the system
is linear, every bus potential of the voltage-source study must then be the
corresponding current-source potential scaled by $I / I_{\text{DC}}$ — a
cheap, exact check that the DC path is doing arithmetic and not something
else. `source_impedance` needs a real part for this: a purely inductive
internal impedance is zero at DC and the division above would not exist.

In [ ]:
current_driven = build("dc_ref", CARSON, CARSON, frequencies=(0.0,))
gi.run_fault(current_driven, fault="F")
u_ref = {bus: value.real for bus, value in epr(current_driven).items()}

thevenin = build("thevenin", CARSON, CARSON, frequencies=(0.0,), voltage_source=True)
gi.run_fault(thevenin, fault="F")
u_th = {bus: value.real for bus, value in epr(thevenin).items()}

z_loop = (u_ref["A"] - u_ref["C"]) / I_DC
i_th = 1000.0 / (0.5 + z_loop)
print(f"loop impedance A->C      : {z_loop:8.5f} Ohm")
print(f"current driven by 1000 V : {i_th:8.4f} A  (behind 0.5 Ohm)")
print()
print("  bus   voltage source / V   scaled current source / V   rel. difference")
print("  " + "-" * 72)
for bus in ("A", "B", "C"):
    scaled = u_ref[bus] * i_th / I_DC
    print(f"  {bus:<5} {u_th[bus]:18.6f}   {scaled:25.6f}   "
          f"{abs(u_th[bus] - scaled) / abs(scaled):15.3e}")

## Summary

| section | claim | reference | result |
|---|---|---|---|
| 1 | `f = 0.1 Hz` is accurate for `I*2*pi*f*L` and wrong for a constant `I*X` | the same study at `f = 0` | 11 digits vs. >50 % error |
| 2 | a finite reactance at DC is dropped and announced | — | one `DCLimitWarning` for ten branches of one type |
| 3 | Carson's earth-return term tends to the DC resistance | the closed-form limit $0$ | exactly $2.5\ \Omega$ |
| 3 | a real part of $-\infty$ is refused | — | `ValueError` |
| 4 | an ideal bond is substituted, not opened | node-merged $2\times2$ system | better than $10^{-7}$ relative, against factors of 14 / 46 / 10 when dropped |
| 5 | the DC bin of a transient run agrees with the stationary one | the same merged reference | better than $10^{-7}$ relative |
| 5 | the state-space solver needs no stand-in | the same merged reference | agrees to rounding |
| 6 | a capacitance stays an open circuit | open-circuit reference | exact, no warning |
| 7 | DC and 50 Hz in one study are independent | an AC-only study | identical bit for bit |
| 7 | a DC voltage source is the equivalent current source | the loop impedance $u_A-u_C$ over $I$ | $\sim10^{-16}$ relative |